In [9]:
import os
import re
import pandas as pd
from glob import glob

# ====== CONFIG ======
base = "/Users/annabelshinichen/Desktop/SchoolWork/MSBMI/Davoli_lab/TCGA/Top_xCell_Continuous_Cytoband_Hits"
cold_files = os.path.join(base, "*_COLD_heatmap_matrix.csv")
hot_files  = os.path.join(base, "*_HOT_heatmap_matrix.csv")

# ========= HELPERS =========
def get_celltype_name(fname: str) -> str:
    """Return just the cell type (drop _COLD/_HOT suffix)."""
    base = os.path.basename(fname).replace("_heatmap_matrix.csv", "")
    if base.endswith("_COLD") or base.endswith("_HOT"):
        return base.rsplit("_", 1)[0]
    return base

def count_hits_per_cytoband_9p(filepaths):
    """
    Build a matrix with ROWS = 9p cytobands, COLS = cell types.
    Each value = number of tumor types that have a hit for that cytoband in that cell type.
    Assumes: rows = tumor types, columns = cytobands (as in your screenshot).
    Treat any non-zero, non-NA entry as a hit.
    """
    out = None
    for fp in filepaths:
        ct = get_celltype_name(fp)
        df = pd.read_csv(fp, index_col=0)

        # some CSVs may have duplicate "Unnamed" columns—drop those
        df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

        # keep only 9p cytobands
        ninep_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("9p")]
        if not ninep_cols:
            continue
        sub = df[ninep_cols]

        # binarize: any non-zero (after coercion) counts as a hit
        sub_num = pd.to_numeric(sub.stack(), errors="coerce").unstack()
        hits = (sub_num.fillna(0) != 0).astype(int)

        # count tumor types (rows) per cytoband (columns)
        counts = hits.sum(axis=0).to_frame(ct)  # index = cytoband, col = this cell type

        # join into output matrix
        if out is None:
            out = counts
        else:
            out = out.join(counts, how="outer")

    if out is None:
        out = pd.DataFrame()  # no matches found
    else:
        out = out.fillna(0).astype(int).sort_index()
    return out

# ======= RUN (9p ONLY) =======
ninep_counts_COLD = count_hits_per_cytoband_9p(cold_files)  # rows = 9p cytobands, cols = cell types
ninep_counts_HOT  = count_hits_per_cytoband_9p(hot_files)

print("=== 9p cytobands (COLD) — counts of tumor types per cell type ===")
print(ninep_counts_COLD)

print("\n=== 9p cytobands (HOT) — counts of tumor types per cell type ===")
print(ninep_counts_HOT)


=== 9p cytobands (COLD) — counts of tumor types per cell type ===
        B_cells  CD4_tcells  CD8_tcells  Dendritic_cells  Endothelium  \
9p11.2        1           0           4                2            0   
9p12          1           0           4                2            0   
9p13.1        3           0           4                2            0   
9p13.2        3           1           4                4            0   
9p13.3        2           0           4                2            0   
9p21.1        3           0           6                2            1   
9p21.2        2           0           5                2            1   
9p21.3        3           1           4                2            2   
9p22.1        2           0           4                2            2   
9p22.2        2           0           4                3            2   
9p22.3        1           0           5                3            2   
9p23          0           0           6                2  

In [15]:
import pandas as pd

# === ARM-LEVEL FIBROBLAST-HIGH ===
arm_path = "/Users/annabelshinichen/Desktop/SchoolWork/MSBMI/Davoli_lab/TCGA/Top_xCell_Continuous_Arm_Hits/Fibroblasts_table.csv"

# Load
arm_df = pd.read_csv(arm_path)
arm_df = arm_df[arm_df["AllHits"].notna()]  # drop NA

# Subset to immune-HOT (i.e., fibroblast high)
arm_hot = arm_df[arm_df["AllHits"].str.contains("ImmuneHot")]

# Count all CNVs (Gain + Loss)
arm_total = len(arm_hot)
arm_gain = arm_hot["AllHits"].str.contains("Gain").sum()
arm_loss = arm_hot["AllHits"].str.contains("Loss").sum()

print("=== Fibroblast-High (HOT) — Arm-Level ===")
print(f"Total SCNAs: {arm_total}")
print(f"  - Gains: {arm_gain}")
print(f"  - Losses: {arm_loss}")

=== Fibroblast-High (HOT) — Arm-Level ===
Total SCNAs: 70
  - Gains: 41
  - Losses: 29


In [16]:
import pandas as pd

# Load your cytoband matrix
cyto_path = "/Users/annabelshinichen/Desktop/SchoolWork/MSBMI/Davoli_lab/TCGA/Top_xCell_Continuous_Cytoband_Hits/Fibroblasts_HOT_heatmap_matrix.csv"
df = pd.read_csv(cyto_path, index_col=0)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Convert all values to numeric (in case some cells are strings)
df = df.apply(pd.to_numeric, errors="coerce")

# Count non-zero gain/loss associations
gain_hits = (df > 0).sum().sum()
loss_hits = (df < 0).sum().sum()

print("=== Fibroblast-High (HOT) — Cytoband-Level (Signed) ===")
print(f"Total Gain-associated cytoband hits: {gain_hits}")
print(f"Total Loss-associated cytoband hits: {loss_hits}")
print(f"Total SCNA hits (gain + loss): {gain_hits + loss_hits}")

=== Fibroblast-High (HOT) — Cytoband-Level (Signed) ===
Total Gain-associated cytoband hits: 715
Total Loss-associated cytoband hits: 921
Total SCNA hits (gain + loss): 1636
